<a href="https://colab.research.google.com/github/GeraldL19/Final-Year-Project-2024/blob/main/SEC_filing_test_19_10_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import libraries
import pandas as pd
import re

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
# nltk imports
from nltk.tokenize import word_tokenize  # tokenize the text == the text is splitted into words in list
from nltk.corpus import stopwords  # this contain common stop words that has no effect in analysis
from nltk.stem import WordNetLemmatizer  # Lemmatization is the process of grouping together the different inflected forms of a word so they can be analyzed as a single item

# download nltk corpus (first time only)
import nltk

nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package basque_grammars to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping grammars/basque_grammars.zip.
[nltk_data]    | Downloading package bcp47 to /root/nltk_data...
[nltk_data]    | Downloading package biocreative_ppi to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   U

True

In [14]:
import numpy as np

In [2]:
!pip install datasets
from datasets import load_dataset

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.6/519.6 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 14.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 31.7 MB/s eta 0:00:00


In [8]:
dataset1 = load_dataset("JanosAudran/financial-reports-sec", "small_full")

Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/200000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [6]:
df = dataset['train'].to_pandas()

In [7]:
df

,cik,sentence,section,labels,filingDate,docID,sentenceID,sentenceCount
0,0000001750,ITEM 1.BUSINESS General AAR CORP. and its subs...,0,"{'1d': 0, '5d': 1, '30d': 0}",2020-07-21,0000001750_10-K_2020,0000001750_10-K_2020_section_1_0,1
1,0000001750,"AAR was founded in 1951, organized in 1955 and...",0,"{'1d': 0, '5d': 1, '30d': 0}",2020-07-21,0000001750_10-K_2020,0000001750_10-K_2020_section_1_1,2
2,0000001750,We are a diversified provider of products and ...,0,"{'1d': 0, '5d': 1, '30d': 0}",2020-07-21,0000001750_10-K_2020,0000001750_10-K_2020_section_1_2,3
3,0000001750,Fiscal 2020 began with strategic initiatives f...,0,"{'1d': 0, '5d': 1, '30d': 0}",2020-07-21,0000001750_10-K_2020,0000001750_10-K_2020_section_1_3,4
4,0000001750,Our momentum from a successful fiscal 2019 car...,0,"{'1d': 0, '5d': 1, '30d': 0}",2020-07-21,0000001750_10-K_2020,0000001750_10-K_2020_section_1_4,5
...,...,...,...,...,...,...,...,...
199995,0000003453,The Company’s operations are vulnerable to dis...,1,"{'1d': 1, '5d': 0, '30d': 1}",2020-02-28,0000003453_10-K_2019,0000003453_10-K_2019_section_1A_57,199996
199996,0000003453,Such events will interfere with the Company’s ...,1,"{'1d': 1, '5d': 0, '30d': 1}",2020-02-28,0000003453_10-K_2019,0000003453_10-K_2019_section_1A_58,199997
199997,0000003453,"In addition, severe weather and natural disast...",1,"{'1d': 1, '5d': 0, '30d': 1}",2020-02-28,0000003453_10-K_2019,0000003453_10-K_2019_section_1A_59,199998
199998,0000003453,These impacts could be particularly acute in c...,1,"{'1d': 1, '5d': 0, '30d': 1}",2020-02-28,0000003453_10-K_2019,0000003453_10-K_2019_section_1A_60,199999


In [27]:
df1 = dataset1['train'].to_pandas()
df1

,cik,sentence,section,labels,filingDate,name,docID,sentenceID,sentenceCount,tickers,exchanges,entityType,sic,stateOfIncorporation,tickerCount,acceptanceDateTime,form,reportDate,returns
0,0000001750,ITEM 1.BUSINESS General AAR CORP. and its subs...,0,"{'1d': 0, '5d': 1, '30d': 0}",2020-07-21,AAR CORP,0000001750_10-K_2020,0000001750_10-K_2020_section_1_0,1,[AIR],[NYSE],operating,3720,DE,1,2020-07-21T17:19:15.000Z,10-K,2020-05-31,{'1d': {'closePriceEndDate': 19.01000022888183...
1,0000001750,"AAR was founded in 1951, organized in 1955 and...",0,"{'1d': 0, '5d': 1, '30d': 0}",2020-07-21,AAR CORP,0000001750_10-K_2020,0000001750_10-K_2020_section_1_1,2,[AIR],[NYSE],operating,3720,DE,1,2020-07-21T17:19:15.000Z,10-K,2020-05-31,{'1d': {'closePriceEndDate': 19.01000022888183...
2,0000001750,We are a diversified provider of products and ...,0,"{'1d': 0, '5d': 1, '30d': 0}",2020-07-21,AAR CORP,0000001750_10-K_2020,0000001750_10-K_2020_section_1_2,3,[AIR],[NYSE],operating,3720,DE,1,2020-07-21T17:19:15.000Z,10-K,2020-05-31,{'1d': {'closePriceEndDate': 19.01000022888183...
3,0000001750,Fiscal 2020 began with strategic initiatives f...,0,"{'1d': 0, '5d': 1, '30d': 0}",2020-07-21,AAR CORP,0000001750_10-K_2020,0000001750_10-K_2020_section_1_3,4,[AIR],[NYSE],operating,3720,DE,1,2020-07-21T17:19:15.000Z,10-K,2020-05-31,{'1d': {'closePriceEndDate': 19.01000022888183...
4,0000001750,Our momentum from a successful fiscal 2019 car...,0,"{'1d': 0, '5d': 1, '30d': 0}",2020-07-21,AAR CORP,0000001750_10-K_2020,0000001750_10-K_2020_section_1_4,5,[AIR],[NYSE],operating,3720,DE,1,2020-07-21T17:19:15.000Z,10-K,2020-05-31,{'1d': {'closePriceEndDate': 19.01000022888183...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,0000003453,The Company’s operations are vulnerable to dis...,1,"{'1d': 1, '5d': 0, '30d': 1}",2020-02-28,"Matson, Inc.",0000003453_10-K_2019,0000003453_10-K_2019_section_1A_57,199996,[MATX],[NYSE],operating,4400,HI,1,2020-02-28T15:00:10.000Z,10-K,2019-12-31,{'1d': {'closePriceEndDate': 31.66852188110351...
199996,0000003453,Such events will interfere with the Company’s ...,1,"{'1d': 1, '5d': 0, '30d': 1}",2020-02-28,"Matson, Inc.",0000003453_10-K_2019,0000003453_10-K_2019_section_1A_58,199997,[MATX],[NYSE],operating,4400,HI,1,2020-02-28T15:00:10.000Z,10-K,2019-12-31,{'1d': {'closePriceEndDate': 31.66852188110351...
199997,0000003453,"In addition, severe weather and natural disast...",1,"{'1d': 1, '5d': 0, '30d': 1}",2020-02-28,"Matson, Inc.",0000003453_10-K_2019,0000003453_10-K_2019_section_1A_59,199998,[MATX],[NYSE],operating,4400,HI,1,2020-02-28T15:00:10.000Z,10-K,2019-12-31,{'1d': {'closePriceEndDate': 31.66852188110351...
199998,0000003453,These impacts could be particularly acute in c...,1,"{'1d': 1, '5d': 0, '30d': 1}",2020-02-28,"Matson, Inc.",0000003453_10-K_2019,0000003453_10-K_2019_section_1A_60,199999,[MATX],[NYSE],operating,4400,HI,1,2020-02-28T15:00:10.000Z,10-K,2019-12-31,{'1d': {'closePriceEndDate': 31.66852188110351...


In [11]:
df1.iloc[0,18]

{'1d': {'closePriceEndDate': 19.010000228881836,
  'closePriceStartDate': 18.190000534057617,
  'endDate': '2020-07-22T00:00:00-04:00',
  'startDate': '2020-07-20T00:00:00-04:00',
  'ret': 0.04507969692349434},
 '5d': {'closePriceEndDate': 17.719999313354492,
  'closePriceStartDate': 18.190000534057617,
  'endDate': '2020-07-27T00:00:00-04:00',
  'startDate': '2020-07-20T00:00:00-04:00',
  'ret': -0.025838438421487808},
 '30d': {'closePriceEndDate': 19.25,
  'closePriceStartDate': 18.190000534057617,
  'endDate': '2020-08-20T00:00:00-04:00',
  'startDate': '2020-07-20T00:00:00-04:00',
  'ret': 0.05827374383807182}}

In [12]:
df1.iloc[199998,18]

{'1d': {'closePriceEndDate': 31.668521881103516,
  'closePriceStartDate': 31.668521881103516,
  'endDate': '2020-03-02T00:00:00-05:00',
  'startDate': '2020-02-27T00:00:00-05:00',
  'ret': 0.0},
 '5d': {'closePriceEndDate': 32.288360595703125,
  'closePriceStartDate': 31.668521881103516,
  'endDate': '2020-03-04T00:00:00-05:00',
  'startDate': '2020-02-27T00:00:00-05:00',
  'ret': 0.019572706893086433},
 '30d': {'closePriceEndDate': 28.569377899169922,
  'closePriceStartDate': 31.668521881103516,
  'endDate': '2020-03-30T00:00:00-04:00',
  'startDate': '2020-02-27T00:00:00-05:00',
  'ret': -0.09786196798086166}}

In [28]:
df1.groupby('sic')['sentenceID'].nunique()

KeyError: ignored

In [31]:
df1.groupby('labels')['sic'].unique()

TypeError: ignored